# Cross-encoder reranking v2 — crash-resilient, and already a strong result

**First run:** `bge-reranker-v2-m3` gave **Darija R@1: 0.575 → 0.800** (+0.225) before the cell crashed on a later model. That single number is already ~15x bigger than anything the loss-function experiments achieved — strong support for "this is a ranking problem, not a representation problem."

**What crashed it:** `gte-multilingual-reranker-base`'s custom attention code hit `CUBLAS_STATUS_NOT_SUPPORTED` on a T4 GPU (likely a bf16 op the GPU doesn't support), inside `.predict()` — outside the original try/except, which only guarded model loading.

**Fixed here:**

- The **entire** rerank call (load + predict) is now guarded — an inference-time crash skips that model instead of ending the run
- **Forces float32** to avoid the bf16 incompatibility
- **Checkpoints after every model** to `rerank_results_partial.pkl` and resumes automatically, so your bge result is never at risk again
- Dropped `jina-reranker-v2` (incompatible custom code) and `gte-multilingual-reranker-base` (the GPU crash) — replaced with `bge-reranker-base` for a size comparison against the v2 model that already worked

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.** No API key.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers transformers accelerate

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    # Candidates passed from retrieval into the reranker. Larger = higher
    # ceiling but slower, since the cross-encoder scores every candidate.
    "rerank_k": 20,

    # Multilingual cross-encoders, strongest first. Any that fails to load is
    # skipped rather than killing the run.
    "rerankers": [
        "BAAI/bge-reranker-v2-m3",                          # already confirmed working, big win
        "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",       # standard CrossEncoder class, lighter, safe
        "BAAI/bge-reranker-base",                            # smaller BGE variant, for a size comparison
        # jina-reranker-v2 and gte-multilingual-reranker-base removed:
        # jina's custom code is incompatible with the installed transformers version;
        # gte's custom attention triggers CUBLAS_STATUS_NOT_SUPPORTED on T4-class GPUs.
        # Both are now caught safely by the guards below if you re-add them.
    ],

    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
    "batch_size": 32,
}

### Load data

In [ ]:
import json, re, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# No training happens here, so the full benchmark is used as the test set.
print(f"Corpus {len(corpus)} | evaluating on all {len(qa)} Wikipedia items")

### BM25 + hybrid retrieval (unchanged from earlier notebooks)

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Stage 1: retrieve candidates, then free the bi-encoder

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("Building index and retrieving candidates...")
bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve_candidates(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = CONFIG["alpha"] * minmax(corpus_emb @ q) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query))
    idx = np.argsort(-s)[:k]
    return [corpus_ids[i] for i in idx]

K = CONFIG["rerank_k"]
candidates = {}
for field in ["msa_query", "darija_query"]:
    candidates[field] = {q["id"]: retrieve_candidates(q[field], K) for q in qa}
    print(f"  {field} done")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### The ceiling: what is the best a reranker could possibly do?

In [ ]:
print("=" * 78)
print(f"CEILING ANALYSIS — Recall@{K} of the retrieval stage")
print("=" * 78)
print("A reranker can only reorder what retrieval already returned. Recall@K is")
print("therefore a hard upper bound on post-rerank Recall@1.\n")

ceiling = {}
for field in ["msa_query", "darija_query"]:
    hits = [int(q["source_chunk_id"] in candidates[field][q["id"]]) for q in qa]
    ceiling[field] = float(np.mean(hits))
    print(f"  {field:<14} Recall@{K} = {ceiling[field]:.3f}   <- reranking ceiling")

print(f"\nCurrent Darija Recall@1 is ~0.575, so the available headroom is roughly")
print(f"{ceiling['darija_query'] - 0.575:.3f} -- far larger than anything the loss-function")
print("experiments moved (+0.015 at best).")

### Baseline (no reranking) for comparison

In [ ]:
def evaluate_order(ordered_ids_by_qid):
    """Per-item hit vectors from an ordered candidate list, for bootstrapping."""
    out = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_ids_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            out[f"R@{k}"].append(1.0 if (pos is not None and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in out.items()}, "MRR": np.array(rr)}

results = {}
for field in ["msa_query", "darija_query"]:
    results[("no_rerank", field)] = evaluate_order(candidates[field])

print("\nBaseline (retrieval order, no reranking):")
for field in ["msa_query", "darija_query"]:
    m = results[("no_rerank", field)]
    print(f"  {field:<14} R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

### Stage 2: cross-encoder reranking

In [ ]:
from sentence_transformers import CrossEncoder
import os, pickle

RESULTS_CACHE = "rerank_results_partial.pkl"
if os.path.exists(RESULTS_CACHE):
    with open(RESULTS_CACHE, "rb") as f:
        results.update(pickle.load(f))
    print(f"Resumed {len(results)} cached result entries.")

def save_results():
    # Checkpoint after every model, not just at the end -- a crash in model N
    # must not cost the results already obtained from models 1..N-1.
    with open(RESULTS_CACHE, "wb") as f:
        pickle.dump(results, f)

def rerank_with(model_name):
    """Score every (query, candidate) pair jointly and reorder.
    The ENTIRE body is guarded, not just model loading: a GPU-specific
    failure inside .predict() (e.g. an unsupported op for this GPU's compute
    capability) must not kill the whole run and lose earlier models' results.
    Forcing float32 sidesteps the most common cause on T4-class GPUs, which
    lack bf16 matmul support that some custom model code assumes by default.
    """
    try:
        ce = CrossEncoder(
            model_name, max_length=512, trust_remote_code=True,
            automodel_args={"torch_dtype": torch.float32},
        )
    except Exception as e:
        print(f"  SKIPPED (load failed) {model_name}: {type(e).__name__}: {str(e)[:150]}")
        return None

    out = {}
    try:
        for field in ["msa_query", "darija_query"]:
            reordered = {}
            for q in qa:
                cands = candidates[field][q["id"]]
                pairs = [(q[field], corpus_map[c]) for c in cands]
                scores = ce.predict(pairs, batch_size=CONFIG["batch_size"], show_progress_bar=False)
                order = np.argsort(-np.asarray(scores))
                reordered[q["id"]] = [cands[i] for i in order]
            out[field] = reordered
            print(f"    {field} reranked")
    except RuntimeError as e:
        print(f"  SKIPPED (inference failed) {model_name}: {type(e).__name__}: {str(e)[:150]}")
        print(f"  (Likely a GPU/dtype incompatibility with this model's custom code, "
              f"not a problem with your data or pipeline.)")
        del ce
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None

    del ce
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

for name in CONFIG["rerankers"]:
    short = name.split("/")[-1]
    if (short, "darija_query") in results:
        print(f"\n=== {short} === (already done, skipping)")
        continue
    print(f"\n=== {short} ===")
    reordered = rerank_with(name)
    if reordered is None:
        continue
    for field in ["msa_query", "darija_query"]:
        results[(short, field)] = evaluate_order(reordered[field])
    save_results()
    m = results[(short, "darija_query")]
    print(f"  Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

### Results table

In [ ]:
rows = []
for (method, field), m in results.items():
    rows.append({
        "method": method, "query": field,
        **{k: float(v.mean()) for k, v in m.items()},
    })
df = pd.DataFrame(rows)
df.to_csv("rerank_results.csv", index=False)

print("\n" + "=" * 78)
print("RESULTS")
print("=" * 78)
pivot = df.pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))

### Improvement over no reranking, with paired bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 78)
print("RERANKING GAIN over retrieval order (paired 95% CI)")
print("=" * 78)
gain_rows = []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    for field in ["msa_query", "darija_query"]:
        if (method, field) not in results:
            continue
        for metric in ["R@1", "MRR"]:
            d, lo, hi = paired(results[(method, field)][metric],
                               results[("no_rerank", field)][metric])
            sig = "yes" if lo > 0 else ("worse" if hi < 0 else "no")
            gain_rows.append({"method": method, "query": field, "metric": metric,
                              "gain": d, "lo": lo, "hi": hi, "significant": sig})
gains = pd.DataFrame(gain_rows)
print(gains.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gains.to_csv("rerank_gains.csv", index=False)

### Effect on the dialect gap, which is what the project is about

In [ ]:
print("\n" + "=" * 78)
print("EFFECT ON THE DIALECT GAP (MSA - Darija)")
print("=" * 78)
gap_rows = []
for method in df.method.unique():
    if (method, "msa_query") not in results:
        continue
    for metric in ["R@1", "MRR"]:
        d, lo, hi = paired(results[(method, "msa_query")][metric],
                           results[(method, "darija_query")][metric])
        gap_rows.append({"method": method, "metric": metric, "gap": d,
                         "lo": lo, "hi": hi,
                         "gap_significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gap_rows).sort_values(["metric", "gap"])
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv("rerank_dialect_gap.csv", index=False)

print("""
Two outcomes are both worth reporting:
  - Gap SHRINKS  -> reranking mitigates dialect mismatch; a practical recommendation
                    that needs no training data at all.
  - Gap PERSISTS -> the penalty is not merely a ranking artefact of the bi-encoder;
                    it survives a much stronger ranking model, which is a stronger
                    claim about the phenomenon than anything measured so far.
""")

### Headline

In [ ]:
best = df[(df["query"] == "darija_query") & (df.method != "no_rerank")]
if len(best):
    top = best.loc[best["R@1"].idxmax()]
    base_r1 = df[(df.method == "no_rerank") & (df["query"] == "darija_query")]["R@1"].iloc[0]
    print("=" * 78)
    print(f"BEST RERANKER: {top['method']}")
    print("=" * 78)
    print(f"  Darija R@1   {base_r1:.3f} -> {top['R@1']:.3f}   ({top['R@1'] - base_r1:+.3f})")
    print(f"  Ceiling (Recall@{K})            {ceiling['darija_query']:.3f}")
    print(f"  Headroom captured              "
          f"{(top['R@1'] - base_r1) / max(ceiling['darija_query'] - base_r1, 1e-9) * 100:.1f}%")
else:
    print("No reranker loaded successfully — check the model names in CONFIG.")

from google.colab import files
files.download("rerank_results.csv")
files.download("rerank_gains.csv")
files.download("rerank_dialect_gap.csv")